In [1]:

import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from torchvision import transforms
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


Using device: cuda


In [2]:

DATA_DIR = "./data"
PCA_COMPONENTS = 35
TEST_SIZE = 0.4
EPOCHS = 50
BATCH_SIZE = 16
USE_PCA = True
LEARNING_RATE = 0.0001


In [3]:

def load_rssi_data():
    files = glob.glob(os.path.join(DATA_DIR, "wifisignal_data_*.csv"))
    data, labels = [], []
    for f in files:
        try:
            df = pd.read_csv(f)
            pkt_cols = [c for c in df.columns if c.startswith("pkt")]
            X = df[pkt_cols].values.astype(float)
            y = df["label"].values
            data.append(X)
            labels.append(y)
        except Exception as e:
            print(f"Error loading {f}: {e}")
    X = np.vstack(data)
    y = np.hstack(labels)
    return X, y


In [4]:

class CNNLSTMModel(nn.Module):
    def __init__(self, input_size, num_classes):
        super(CNNLSTMModel, self).__init__()
        self.conv1 = nn.Conv1d(1, 64, kernel_size=3)
        self.pool = nn.MaxPool1d(2)
        self.lstm = nn.LSTM(64, 64, batch_first=True)
        self.fc1 = nn.Linear(64, 64)
        self.fc2 = nn.Linear(64, num_classes)

    def forward(self, x):
        x = x.unsqueeze(1)  # (B, 1, L)
        x = F.relu(self.conv1(x))
        x = self.pool(x)
        x = x.permute(0, 2, 1)  # for LSTM: (B, T, F)
        _, (hn, _) = self.lstm(x)
        x = F.relu(self.fc1(hn[-1]))
        x = self.fc2(x)
        return x


In [5]:

X, y = load_rssi_data()
le = LabelEncoder()
y_enc = le.fit_transform(y)
num_classes = len(le.classes_)
X_train, X_test, y_train, y_test = train_test_split(X, y_enc, test_size=TEST_SIZE, random_state=42, stratify=y_enc)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

if USE_PCA:
    pca = PCA(n_components=PCA_COMPONENTS)
    X_train_processed = pca.fit_transform(X_train_scaled)
    X_test_processed = pca.transform(X_test_scaled)
    input_size = PCA_COMPONENTS
else:
    X_train_processed = X_train_scaled
    X_test_processed = X_test_scaled
    input_size = X.shape[1]

train_ds = TensorDataset(torch.tensor(X_train_processed, dtype=torch.float32),
                         torch.tensor(y_train, dtype=torch.long))
test_ds = TensorDataset(torch.tensor(X_test_processed, dtype=torch.float32),
                        torch.tensor(y_test, dtype=torch.long))

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE)

model = CNNLSTMModel(input_size=input_size, num_classes=num_classes).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
criterion = nn.CrossEntropyLoss()

train_loss_log, test_acc_log = [], []
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        preds = model(xb)
        loss = criterion(preds, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    train_loss_log.append(total_loss / len(train_loader))

    model.eval()
    correct = total = 0
    with torch.no_grad():
        for xb, yb in test_loader:
            xb, yb = xb.to(device), yb.to(device)
            preds = model(xb)
            pred_labels = preds.argmax(dim=1)
            correct += (pred_labels == yb).sum().item()
            total += yb.size(0)
    test_acc = correct / total
    test_acc_log.append(test_acc)
    print(f"Epoch {epoch+1}, Train Loss: {total_loss:.4f}, Test Acc: {test_acc:.4f}")


KeyboardInterrupt: 

In [ ]:

plt.plot(train_loss_log, label='Train Loss')
plt.plot(test_acc_log, label='Test Accuracy')
plt.title("Training Progress")
plt.legend(); plt.grid(True); plt.show()


In [ ]:

model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.to(device)
        preds = model(xb).argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(yb.numpy())

cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=le.classes_, yticklabels=le.classes_)
plt.xlabel("Predicted"); plt.ylabel("True"); plt.title("Confusion Matrix")
plt.show()

report = classification_report(all_labels, all_preds, target_names=le.classes_)
print(report)
